# Fight des IA — Arène des algos J3

Quatre problèmes, un leaderboard final.

## Phase 0 — Mise en route

In [ ]:
%pip install numpy pandas matplotlib scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
    accuracy_score,
    f1_score,
    classification_report,
    silhouette_score,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_california_housing

In [ ]:
## Phase A — Prédire les prix immobiliers (régression)

In [ ]:
def charger_immobilier():
    """Charge California Housing, renvoie X, y.

    Doit afficher : nombre de lignes, nombre de variables, et l'unité de la cible.
    """
    data = fetch_california_housing()
    X = data.data
    y = data.target

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print("Cible = prix médian en centaines de milliers de $")

    return X, y

In [ ]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.

    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    return {
        "r2": r2_score(y_test, y_pred),
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
    }

In [ ]:
X_immo, y_immo = charger_immobilier()

X_train, X_test, y_train, y_test = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for nom, modele in [
    ("LinearRegression", LinearRegression()),
    ("RandomForest", RandomForestRegressor(random_state=42)),
]:
    scores = evaluer_regression(modele, X_train_s, X_test_s, y_train, y_test)
    print(f"{nom:<18} : R2={scores['r2']:.2f}  MAE={scores['mae']:.2f}  RMSE={scores['rmse']:.2f}")

In [ ]:
print("=== Cas limite : 100 lignes seulement ===")
X_petit = X_immo[:100]
y_petit = y_immo[:100]
X_tr, X_te, y_tr, y_te = train_test_split(X_petit, y_petit, test_size=0.2, random_state=42)
sc = StandardScaler()
scores_petit = evaluer_regression(
    LinearRegression(),
    sc.fit_transform(X_tr), sc.transform(X_te), y_tr, y_te
)
print(f"R2 avec 100 lignes : {scores_petit['r2']:.2f} — s'effondre : pas assez de données pour apprendre.")

In [ ]:
print("=== Cas adversarial : quartier fictif (revenu=0, 9000 habitants) ===")
quartier_fictif = np.array([[
    0,  # MedInc
    np.median(X_immo[:, 1]),
    np.median(X_immo[:, 2]),
    np.median(X_immo[:, 3]),
    9000,  # Population
    np.median(X_immo[:, 5]),
    np.median(X_immo[:, 6]),
    np.median(X_immo[:, 7]),
]])

modele_lr = LinearRegression()
modele_lr.fit(X_train_s, y_train)
prix_pred = modele_lr.predict(scaler.transform(quartier_fictif))[0]

print(f"Prix prédit : {prix_pred:.2f} (centaines de milliers $)")
print("→ Hors plage d'entraînement : en prod, il faudrait rejeter ou borner cette prédiction.")